In [1]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
kagglehub.competition_download(
    'kmu-rec-sys-26-rating-prediction',
    output_dir = "dataset"
)

100%|██████████| 5.76M/5.76M [00:00<00:00, 125MB/s]

Extracting files...


'dataset'

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
!head dataset/train_small.csv

userId,movieId,rating,date
64777,16043,5.0,1997-07-01
64777,5112,2.0,1997-07-02
64777,24688,4.0,1997-07-02
64777,16639,4.0,1997-07-01
64777,13084,4.0,1997-07-02
64777,15142,4.0,1997-07-03
64777,5741,3.0,1997-07-03
64777,25503,3.0,1997-07-03
64777,2780,4.0,1997-07-01


In [5]:
!head dataset/test_small.csv

userId,movieId
64777,12355
64777,1864
64777,7104
64777,22572
64777,6751
42001,994
42001,11270
42001,23855
42001,19015


##데이터준비

In [6]:
import csv
import torch
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

userIds, itemIds = {}, {}
users, items, ratings, dates = [], [], [], []

with open('dataset/train_small.csv',"r") as f:
  reader = csv.reader(f)
  next(reader)
  for userId, itemId, rating, date in reader:
    if userId not in userIds:
      userIds[userId] = len(userIds)
    if itemId not in itemIds:
      itemIds[itemId] = len(itemIds)

    users.append(userIds[userId])
    items.append(itemIds[itemId])
    ratings.append(float(rating))
    dates.append(datetime.strptime(date, "%Y-%m-%d"))

users = torch.tensor(users).to(device)
items = torch.tensor(items).to(device)
ratings = torch.tensor(ratings).to(device)

tmin = min(dates) # dates 날짜 중 가장 빠른 날짜
dates = [(t-tmin).days for t in dates]
dates = torch.tensor(dates, device=device)

In [7]:
tmin

datetime.datetime(1996, 12, 12, 0, 0)

##데이터셋 분할

In [8]:
n_train=int(len(ratings)*0.9)
indices=torch.randperm(len(ratings))
train_indices=indices[:n_train]
valid_indices=indices[n_train:]

users_valid=users[valid_indices]
items_valid=items[valid_indices]
ratings_valid=ratings[valid_indices]
dates_valid=dates[valid_indices]

users=users[train_indices]
items=items[train_indices]
ratings=ratings[train_indices]
dates=dates[train_indices]

### Latent Factor Model

In [13]:
n_users, n_items = len(userIds), len(itemIds)

mean = ratings.mean()
user_bias = torch.zeros(n_users, requires_grad=True, device=device)
item_bias = torch.zeros(n_items, requires_grad=True, device=device)
user_emb = torch.normal(0, 0.1,size=(n_users,10),requires_grad=True, device=device)
item_emb = torch.normal(0, 0.1,size=(n_items,10),requires_grad=True, device=device)

def predict(users, items):
  h = mean + user_bias[users] + item_bias[items]
  h += (user_emb[users]*item_emb[items]).sum(dim=1)
  return h

optim = torch.optim.Adam([user_bias, item_bias, user_emb, item_emb],
                         lr = 0.01, weight_decay=0.00001)

mse = torch.nn.MSELoss()

for epoch in range(1000):
  h = predict(users ,items)
  cost = mse(h, ratings)

  optim.zero_grad()
  cost.backward()
  optim.step()

  if epoch % 100 == 0:
    h_valid = predict(users_valid, items_valid)
    cost_valid = mse(h_valid, ratings_valid)

    print(f"epoch: {epoch}, train mse:{cost.item()}, test mse: {cost_valid.item()}")


epoch: 0, train mse:1.1054530143737793, test mse: 1.0939778089523315
epoch: 100, train mse:0.4954456388950348, test mse: 0.6750631928443909
epoch: 200, train mse:0.46724945306777954, test mse: 0.6582107543945312
epoch: 300, train mse:0.46050429344177246, test mse: 0.6528722643852234
epoch: 400, train mse:0.4576583802700043, test mse: 0.6491480469703674
epoch: 500, train mse:0.4562136232852936, test mse: 0.6464130282402039
epoch: 600, train mse:0.4554114043712616, test mse: 0.6445986032485962
epoch: 700, train mse:0.4549148976802826, test mse: 0.6438603401184082
epoch: 800, train mse:0.4546240568161011, test mse: 0.6439902186393738
epoch: 900, train mse:0.4544607996940613, test mse: 0.644505500793457


##Temporal Item Bias (binning)

In [14]:
n_bins = 30
tmax = max(dates.max(), dates_valid.max())
date_bins = (dates*n_bins//(tmax+1))
date_bins_valid = (dates_valid*n_bins // (tmax+1))


# Temporal Item Bias
item_bin_bias = torch.zeros(n_items, n_bins, device=device, requires_grad=True)

def predict(users, items, date_bins):
    h = mean + user_bias[users] + item_bias[items]
    h += (user_emb[users] * item_emb[items]).sum(dim=1)
    h += item_bin_bias[items, date_bins]

    return h

##Temporal User Bias
Drifting bias

In [15]:
tu = users.bincount(dates)/users.bincount()
dev = (dates-tu[users]).sign() * ((dates - tu[users]).abs()**0.4)
dev_valid = (dates_valid - tu[users_valid]).sign()*((dates_valid - tu[users_valid]).abs()**0.4)

# Temporal User Bias
dev_bias = torch.zeros(n_users, device=device, requires_grad=True)

def predict(users, items, date_bins, dev):
    h = mean + user_bias[users] + item_bias[items]
    h += (user_emb[users] * item_emb[items]).sum(dim=1)
    h += item_bin_bias[items, date_bins]
    h += dev_bias[users] * dev[users]

    return h


Per-day user bias

In [17]:
all_users=torch.cat((users,users_valid))
all_dates=torch.cat((dates,dates_valid))
_,all_userdates=torch.stack((all_users,all_dates)).T.unique(dim=0,return_inverse=True)
userdates=all_userdates[:len(users)]
userdates_valid=all_userdates[len(users):]
n_userdates=all_userdates.max()+1


# Per-day User Bias
perday_bias=torch.zeros(n_userdates,requires_grad=True,device=device)

def predict(users,items,date_bins,dev,userdates):
  h= mean + user_bias[users]+item_bias[items]
  h+=(user_emb[users]*item_emb[items]).sum(dim=1)
  h+=item_bin_bias[items,date_bins]
  h+=dev_bias[users]*dev[users]
  h+=perday_bias[userdates]
  return h